In [176]:
import pyarrow.parquet as pq
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import math
from collections import defaultdict

# INITS

In [177]:
def init_baseline_model(behaviors, history):
    """
    init_predict(dataset) dataset is from history data
    this is the learning phase

    my code needs history data to learn
    """
    # We want information about the articles that were clicked
    # article_ids_scroll_percentages = {}
    # article_ids_read_times = {}
    
    # def build_scroll_percentage_and_read_time(row, article_ids_scroll_percentages, article_ids_read_times):
    #     for article_id, scroll_percentage_fixed, read_time_fixed in zip(row['article_id_fixed'], row['scroll_percentage_fixed'], row['read_time_fixed']):
    #         # print(article_ids_scroll_percentages.get(article_id, []))
    #         if article_id not in article_ids_scroll_percentages:
    #             article_ids_scroll_percentages[article_id] = []
    #         if article_id not in article_ids_read_times:
    #             article_ids_read_times[article_id] = []

    #         article_ids_read_times[article_id].append(read_time_fixed)
    #         article_ids_scroll_percentages[article_id].append(scroll_percentage_fixed)

    #         # article_ids_scroll_percentages[article_id] = article_ids_scroll_percentages.get(article_id, []).append(row['scroll_percentage_fixed'])
    #         # article_ids_read_times[article_id] = article_ids_read_times.get(article_id, []).append(row['read_time_fixed'])      

    # history.apply(build_scroll_percentage_and_read_time, axis=1, args=(article_ids_scroll_percentages, article_ids_read_times))
    
    # we compute both clicked_sum and inview_sum for each article_id
    article_ids_clicked_sum = {}
    for i in behaviors['article_ids_clicked']:
        for j in i:
            article_ids_clicked_sum[j] = article_ids_clicked_sum.get(j, 0) + 1

    article_ids_inview_sum = {}
    for i in behaviors['article_ids_inview']:
        for j in i:
            article_ids_inview_sum[j] = article_ids_inview_sum.get(j, 0) + 1

    # this way we can compute the click rate for each article_id and sort them
    efficiency = pd.DataFrame({'clicked': article_ids_clicked_sum, 'inview': article_ids_inview_sum})
    efficiency['article_id'] = efficiency.index
    efficiency.reset_index(drop=True, inplace=True)
    efficiency['clicked'] = efficiency['clicked'].fillna(0)

    efficiency['click_rate'] = efficiency['clicked'] / efficiency['inview']
    
    efficiency = efficiency.sort_values(by='click_rate', ascending=False)

    # we can also comute the average scroll percentage and read time for each article_id
    # article_ids_scroll_percentage_mean = {}
    # for article_id, scroll_percentages in article_ids_scroll_percentages.items():
    #     article_ids_scroll_percentage_mean[article_id] = np.mean(scroll_percentages)
    # article_ids_read_time_mean = {}
    # for article_id, read_times in article_ids_read_times.items():
    #     article_ids_read_time_mean[article_id] = np.mean(read_times)

    # interest = pd.DataFrame({'scroll_percentage': article_ids_scroll_percentage_mean, 'read_time': article_ids_read_time_mean})
    # interest['article_id'] = interest.index
    # interest.reset_index(drop=True, inplace=True)
    
    # efficiency = pd.merge(efficiency, interest, on='article_id', how='outer')
    # efficiency.fillna(0, inplace=True)
    return efficiency

In [178]:
def init_content_model(articles, history):
    """
    Prepares TF-IDF article matrix and user profiles based on their reading history.
    """

    # Combine article text fields safely
    articles['text'] = (
        articles['title'].fillna('') + ' ' +
        articles['subtitle'].fillna('') + ' ' +
        articles['body'].fillna('')
    )

    # Build the article to index mapping
    article_to_index = pd.Series(articles.index, index=articles['article_id']).drop_duplicates()

    # Define Danish stopwords
    danish_stopwords = [
        "og", "i", "det", "er", "som", "på", "de", "en", "til", "med", "at", "for", "der", "af", "han"
    ]

    # Create TF-IDF vectorizer and transform the article texts
    tfidf = TfidfVectorizer(stop_words=danish_stopwords, min_df=2, max_df=0.9)
    article_matrix = tfidf.fit_transform(articles['text'])

    # Prepare user profiles efficiently
    user_profiles = {}

    # Flatten history: explode article_id_fixed (each article_id separately)
    history_exploded = history.explode('article_id_fixed').dropna(subset=['article_id_fixed']).copy()
    history_exploded['index'] = history_exploded['article_id_fixed'].map(article_to_index)

    # Group by user and collect indices
    user_indices = history_exploded.dropna(subset=['index']).groupby('user_id')['index'].apply(list)

    for user_id, indices in user_indices.items():
        user_profile = article_matrix[indices].mean(axis=0)
        user_profiles[user_id] = np.asarray(user_profile).ravel()

    return user_profiles, article_matrix, article_to_index


In [179]:
def init_collab_model(behaviors, history):
    """
    Initialize collaborative filtering model:
    Counts individual item occurrences and item co-occurrences.
    """

    user_items = defaultdict(set)

    # Preprocess behaviors (single articles)
    behaviors = behaviors.dropna(subset=['article_id'])
    for u, art in zip(behaviors['user_id'], behaviors['article_id']):
        user_items[u].add(str(art))

    # Preprocess history (multiple articles per row)
    history = history.dropna(subset=['article_id_fixed'])
    for u, arts in zip(history['user_id'], history['article_id_fixed']):
        if isinstance(arts, (list, set, tuple)):
            for art in arts:
                user_items[u].add(str(art))

    # Initialize counts
    item_count = defaultdict(int)
    co_count = defaultdict(lambda: defaultdict(int))

    for arts in user_items.values():
        arts = list(arts)
        n = len(arts)
        if n == 0:
            continue
        
        # Update item counts
        for art in arts:
            item_count[art] += 1

        # Update co-occurrence counts
        for idx in range(n):
            for jdx in range(idx + 1, n):
                i, j = arts[idx], arts[jdx]
                co_count[i][j] += 1
                co_count[j][i] += 1

    return item_count, co_count


# SCORES

In [180]:
# efficiency = None
# user_profiles = None
# article_matrix = None
# article_to_index = None
# item_count = None
# co_count = None
# history = None

In [181]:
def baseline_model_score(article_id, efficiency=efficiency):
    """
    needs efficiency
    """
    if article_id not in efficiency['article_id'].values:
        # print(f"Article ID {article_id} not found in efficiency data.")
        return 0.0
    else:
        return efficiency[efficiency['article_id'] == article_id]['click_rate'].values[0]

In [182]:
def content_model_score(article_id, user_id, user_profiles=user_profiles, article_matrix=article_matrix, article_to_index=article_to_index):
    """
    Calcule un score de similarité entre un user_id et un article_id
    """
    if user_id not in user_profiles:
        # print(f"User ID {user_id} not found in user_profiles.")
        return 0.0  # Pas de profil pour cet utilisateur

    user_profile = user_profiles[user_id]

    if article_id not in article_to_index.index:
        # print(f"Article ID {article_id} not found in article_to_index.")
        return 0.0  # Pas d'article connu

    article_idx = article_to_index[article_id]

    article_vector = article_matrix[article_idx]

    # Calcul de la similarité cosine
    sim_score = cosine_similarity(user_profile.reshape(1, -1), article_vector.reshape(1, -1)).flatten()[0]

    return sim_score


In [183]:
def collab_model_score(article_id, user_id, history=history, co_count=co_count, item_count=item_count):
    user_id = int(user_id)
    
    user_articles = history[history['user_id'] == user_id]['article_id_fixed'].explode().unique()

    if len(user_articles) == 0:
        return 0.0

    i = str(article_id)
    if i not in co_count:
        return 0.0

    score = 0.0
    norm_i = math.sqrt(item_count[i])

    for user_article in user_articles:
        user_article = str(user_article)
        if user_article in co_count[i]:
            cij = co_count[i][user_article]
            norm_j = math.sqrt(item_count[user_article])
            score += cij / (norm_i * norm_j)

    # Normalize by number of articles read
    # score /= len(user_articles)

    # Optional: squash with tanh to keep score between [-1, 1]
    score = np.tanh(score)

    return score


In [184]:
def hybrid_model_score(article_id, user_id):
    baseline_score = baseline_model_score(article_id)
    content_score = content_model_score(article_id, user_id)
    collab_score = collab_model_score(article_id, user_id)

    w1, w2, w3 = 0.2, 0.4, 0.4
    if history[history['user_id'] == user_id].empty:
        # Si l'utilisater n'a pas d'historique, ne pas prendre en compte le score de content base
        w2 = 0
    if efficiency[efficiency['article_id'] == article_id].empty or efficiency[efficiency['article_id'] == article_id]['inview'].values[0] == 0:
        # Si l'article n'a pas été vu, ne pas prendre en compte le score de baseline
        w1, w3 = 0, 0
    
    weight_sum = w1 + w2 + w3
    if weight_sum == 0:
        # Si tous les poids sont nuls, retourner 0
        return 0.0
    # Normaliser les poids
    w1, w2, w3 = w1 / weight_sum, w2 / weight_sum, w3 / weight_sum
    # print(f"w baseline: {w1}, w content: {w2}, w collab: {w3}")

    # normalize if needed
    #baseline_score = normalize(baseline_score)
    #content_score = normalize(content_score)
    #collab_score = normalize(collab_score)

    final_score = w1 * baseline_score + w2 * content_score + w3 * collab_score
    # print(f"User id: {user_id}, article id: {article_id}, Final score: {final_score}")
    return final_score


# PREDICT

In [185]:
def predict(row):
    """
    This is the prediction function
    takes the list of the articles_ids_inviews and returns the article_id to recommend
    """
    user_id = row['user_id']
    article_ids_inviews = row['article_ids_inview']
    best_article_id = None
    best_score = float('-inf')
    
    for article_id in article_ids_inviews:
        # we compute the score for each article_id
        score = hybrid_model_score(article_id , user_id)
        # print(f"Article ID: {article_id}, Score: {score}")
        
        # update the best article if the current score is higher
        if score > best_score:
            best_score = score
            best_article_id = article_id
    
    return best_article_id

In [186]:
def predict_ranked(row):
    """
    Predict and rank all article_ids_inview for the user.
    Returns a list of article_ids sorted by their predicted score.
    """
    user_id = row['user_id']
    article_ids_inviews = row['article_ids_inview']
    
    article_scores = []
    
    for article_id in article_ids_inviews:
        score = hybrid_model_score(article_id , user_id)
        article_scores.append((article_id, score))
    
    # Sort articles by descending score
    article_scores.sort(key=lambda x: x[1], reverse=False)
    # Return the list of article_ids sorted
    ranked_articles = [article_id for article_id, score in article_scores]
    
    return ranked_articles


# MAIN

In [187]:
articles_path = os.path.join('datas', 'ebnerd_demo', 'articles.parquet')
articles = pq.read_table(articles_path).to_pandas()

history_path = os.path.join('datas', 'ebnerd_demo', 'train', 'history.parquet')
history = pq.read_table(history_path).to_pandas()

behaviors_path = os.path.join('datas', 'ebnerd_demo', 'train', 'behaviors.parquet')
behaviors = pq.read_table(behaviors_path).to_pandas()

# ACCURACY
We will use history to train the models then check the accuracy on behaviors with a window accuracy method.

## WINDOW ACCURACY SIMPLE

In [188]:
data_start = behaviors['impression_time'].min()
data_end = behaviors['impression_time'].max()
data_start, data_end # the time range of the behaviors

(Timestamp('2023-05-18 07:00:03'), Timestamp('2023-05-25 06:59:52'))

In [189]:
def slide(data, window_size, slide_size):
    start = data_start
    end = start + window_size
    while end <= data_end:
        yield data[(data['impression_time'] >= start) & (data['impression_time'] < end)], start, end
        start += slide_size
        end = start + window_size

In [190]:
for window, start, end in slide(behaviors, window_size=pd.Timedelta(days=3), slide_size=pd.Timedelta(days=1)):
    # we split the window into training and test set
    splitting_date = end - pd.Timedelta(days=1)
    # the first days are used for training and the last day for testing
    training = window[window['impression_time'] < splitting_date].copy()
    test = window[window['impression_time'] >= splitting_date].copy()

    # we train the models
    print(f"Training the model for the window {start} to {end}")
    print("Training the baseline model")
    efficiency = init_baseline_model(training, history)
    print("Training the content model")
    user_profiles, article_matrix, article_to_index = init_content_model(articles, history)
    print("Training the collaborative model")
    item_count, co_count = init_collab_model(training, history)

    print(f"Testing the model for the window {start} to {end}")
    # we test the model
    test.loc[:,'recommended_article_id'] = test.apply(predict, axis=1)
    # we compute the accuracy
    accuracy = (test['recommended_article_id'] == test['article_id']).dropna().mean()
    print(f"accuracy for the window {start} to {end} is {accuracy}")

Training the model for the window 2023-05-18 07:00:03 to 2023-05-21 07:00:03
Training the baseline model
Training the content model
Training the collaborative model
Testing the model for the window 2023-05-18 07:00:03 to 2023-05-21 07:00:03
accuracy for the window 2023-05-18 07:00:03 to 2023-05-21 07:00:03 is 0.023543990086741014
Training the model for the window 2023-05-19 07:00:03 to 2023-05-22 07:00:03
Training the baseline model
Training the content model
Training the collaborative model
Testing the model for the window 2023-05-19 07:00:03 to 2023-05-22 07:00:03
accuracy for the window 2023-05-19 07:00:03 to 2023-05-22 07:00:03 is 0.015763829177414732
Training the model for the window 2023-05-20 07:00:03 to 2023-05-23 07:00:03
Training the baseline model
Training the content model
Training the collaborative model
Testing the model for the window 2023-05-20 07:00:03 to 2023-05-23 07:00:03
accuracy for the window 2023-05-20 07:00:03 to 2023-05-23 07:00:03 is 0.020838828805064626
Trai

## WINDOW ACCURACY RANKING

In [191]:
def ranking_score(ranked_articles, true_article_id):
    """
    Computes a reciprocal rank score.
    Higher is better. 1.0 = best (true article ranked 1st).
    """

    if pd.isna(true_article_id):
        return np.nan  # Return NaN if true_article_id is NaN

    true_article_id = int(true_article_id)  # Convert to int if not NaN

    if true_article_id not in ranked_articles:
        # print(f"Article ID {true_article_id} not found in ranked articles.")
        # print(f"Ranked articles: {ranked_articles}")
        return np.nan

    rank = ranked_articles.index(true_article_id) + 1  # +1 because ranks start at 1
    return 1.0 / rank


In [192]:
for window, start, end in slide(behaviors, window_size=pd.Timedelta(days=3), slide_size=pd.Timedelta(days=1)):
    splitting_date = end - pd.Timedelta(days=1)
    training = window[window['impression_time'] < splitting_date].copy()
    test = window[window['impression_time'] >= splitting_date].copy()

    # Train the models
    print(f"Training the model for the window {start} to {end}")
    print("Training the baseline model")
    efficiency = init_baseline_model(training, history)
    print("Training the content model")
    user_profiles, article_matrix, article_to_index = init_content_model(articles, history)
    print("Training the collaborative model")
    item_count, co_count = init_collab_model(training, history)

    print(f"Testing the model for the window {start} to {end}")
    # Predict rankings
    test.loc[:, 'ranked_articles'] = test.apply(predict_ranked, axis=1)

    # Compute ranking scores
    test['ranking_score'] = test.apply(lambda row: ranking_score(row['ranked_articles'], row['article_id']), axis=1)
    
    mean_ranking_score = test['ranking_score'].dropna().mean()
    print(f"Mean reciprocal rank (MRR) for the window {start} to {end} is {mean_ranking_score:.4f}")


Training the model for the window 2023-05-18 07:00:03 to 2023-05-21 07:00:03
Training the baseline model
Training the content model
Training the collaborative model
Testing the model for the window 2023-05-18 07:00:03 to 2023-05-21 07:00:03
Mean reciprocal rank (MRR) for the window 2023-05-18 07:00:03 to 2023-05-21 07:00:03 is 0.2184
Training the model for the window 2023-05-19 07:00:03 to 2023-05-22 07:00:03
Training the baseline model
Training the content model
Training the collaborative model
Testing the model for the window 2023-05-19 07:00:03 to 2023-05-22 07:00:03
Mean reciprocal rank (MRR) for the window 2023-05-19 07:00:03 to 2023-05-22 07:00:03 is 0.2120
Training the model for the window 2023-05-20 07:00:03 to 2023-05-23 07:00:03
Training the baseline model
Training the content model
Training the collaborative model
Testing the model for the window 2023-05-20 07:00:03 to 2023-05-23 07:00:03
Mean reciprocal rank (MRR) for the window 2023-05-20 07:00:03 to 2023-05-23 07:00:03 is